This file contains code related to the following operations -
1. Extraction of reviewer-centric features
2. Extraction of user cluster id feature
3. Merging of all features in a single dataframe and exporting to a csv file
4. Extracting labels and exporting to a separate csv file
5. Model training - Random Classifier - Random Sampling
6. Model training - Random Classifier - Stratifiedkfold Sampling
7. Model training - Naive Bayes Classifier - Stratifiedkfold Sampling

In [ ]:
import numpy as np
import pandas as pd
# from nltk import bigrams,trigrams
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from textblob import TextBlob
from sklearn.decomposition import TruncatedSVD
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
from nltk.stem import PorterStemmer
import re

In [ ]:
#read reviews, remove stop words and perform stemming

file1 = open("reviewContent")
datareview = []
all_stop_words = stopwords.words('english')
st = PorterStemmer()
for row in file1:
    text = re.split(r'\t+', row)
    row = re.sub('[^a-zA-Z0-9 \n\..]', '', text[-1])
    row = ' '.join([word for word in row.split() if word not in all_stop_words])
    row = ' '.join([st.stem(word) for word in row.split()])
    words = row.split()
    datareview.append(len(words))


file1.close()
print(len(datareview))
for i in range(0, 5):
  print(datareview[i])

In [ ]:
#read metadata file and extract date, userid, productid, rating and label for each review

file2 = open("metadata")
datauser = []

for row in file2:
    tr = re.split(r'\t+', row)
    temp = []
    #date
    temp.append(tr[4])
    #userid
    temp.append(tr[0])
    #productid
    temp.append(tr[1])
    #rating
    temp.append(tr[2])
    #label
    if tr[3] == "1":
        temp.append(0)
    else:
        temp.append(1)
    datauser.append(temp)
    
file2.close()
print(len(datauser))
for i in range(0, 5):
  print(datauser[i])

In [ ]:
#feature extraction - user features

userlist = {}
#["date", "userid", "productid", "rating", "label"]
for i, j in zip(datareview, datauser):
  userid = j[1]
  tr = j
  tr.append(i)
  if userid not in userlist:
    userlist[userid] = []
  userlist[userid].append(tr)

print(len(userlist))
print(userlist.keys())

In [ ]:
#feature extraction - user features

userfeatures = []
#["date", "userid", "productid", "rating", "label", "no of words"]
for userid in userlist.keys():
  tr = []
  #user id
  tr.append(userid)
  total, pos, neg = 0, 0, 0
  for i in userlist[userid]:
    total += int(i[-1])
    if float(i[3]) >= 3:
      pos += 1
    else:
      neg += 1
  #average number of words
  tr.append((total / len(userlist[userid])))
  #total number of reviews
  tr.append(len(userlist[userid]))
  #is singleton
  if len(userlist[userid]) > 1:
    tr.append(0)
  else:
    tr.append(1)
  #ratio of positive reviews (> 3)
  tr.append(pos / len(userlist[userid]))
  #ratio of negative reviews (< 3)
  tr.append(neg / len(userlist[userid]))

  date = {}
  for i in userlist[userid]:
    if i[0] not in date:
      date[i[0]] = 0
    date[i[0]] += 1
  #average number of reviews per day
  tr.append(sum(date.values()) / len(date))
  #maximum number of reviews per day
  tr.append(date[max(date, key=date.get)])

  userfeatures.append(tr)

print(len(userfeatures))

finaldata = np.array(userfeatures)
np.savetxt('user.csv', finaldata, delimiter=',', fmt='%s', header="UserID, AvgNoOfWords, TotalReviews, IsSingleton, PosReviewRatio, NegReviewRatio, AvgNoOfReviews, MaxNoOfReviews")

In [ ]:
#feature extraction - user features

import datetime
from datetime import date

def calculateDateVariance(dates):
  diffs = []
  dt = []
  for i in range(0, len(dates)):
    dt.append(dates[i].split("\n")[0])
  #print(dt)
  dates = [datetime.datetime.strptime(ts, "%Y-%m-%d") for ts in dt]
  dates.sort()
  for i in range(1, len(dates)):
    diffs.append(calculateDateDifference(dates[i], dates[i - 1]))
  return np.var(diffs)

def calculateDateDifference(date1, date2):
  delta = date1 - date2
  #print(delta.days)
  return abs(delta.days)

#test = ['2014-12-08\n', '2013-05-16\n','2013-07-01\n', '2011-07-28\n','2010-11-01\n']
#print(calculateDateVariance(test))

In [ ]:
#feature extraction - user features

datevariance = []

for userid in userlist.keys():
  dates = []
  for i in userlist[userid]:
    dates.append(i[0])
  datevariance.append(calculateDateVariance(dates))

print(len(datevariance))
print(datevariance)
np.savetxt('userdatevar.csv', datevariance, delimiter=',', fmt='%s', header="DateVariance")

In [ ]:
#feature extraction - clustering data
import datetime
from datetime import date

file2 = open("metadata")
datacluster = []

for row in file2:
    tr = re.split(r'\t+', row)
    temp = []
    date = tr[4].split("\n")[0]
    date = datetime.datetime.strptime(date, "%Y-%m-%d")
    a_timedelta = date - datetime.datetime(1900, 1, 1)
    #date
    temp.append(a_timedelta.total_seconds())
    #userid
    temp.append(int(tr[0]))
    #productid
    temp.append(int(tr[1]))
    datacluster.append(temp)
    
file2.close()
print(len(datacluster))
for i in range(0, 5):
  print(datacluster[i])

In [ ]:
#export clustering features - date, userid, productid

np.savetxt('datacluster.csv', datacluster, delimiter=',', fmt='%s', header="Date, UserID, ProductID")

In [ ]:
#clustering - extract user cluster id - date as date value
from sklearn.cluster import KMeans
df = pd.read_csv('datacluster.csv')
print(df)

cost = []
for n in range (100, 1000, 100):
  kmeans = KMeans(init="random", n_clusters=n, n_init=10, max_iter=300, random_state=42).fit(df)
  y = kmeans.predict(df)
  cost.append(kmeans.inertia_)
  print(y)

y = np.array([i for i in range(100,1000,100)])
plt.plot(y,cost)



#!pip install kmodes

In [ ]:
#clustering - extract user cluster id - date in seconds
from sklearn.cluster import KMeans
df = pd.read_csv('datacluster_seconds.csv')
print(df)

kmeans = KMeans(init="random", n_clusters=1000, n_init=10, max_iter=300, random_state=42).fit(df)
y = kmeans.predict(df)
np.savetxt('kmeans_cluster_1000.csv', y, delimiter=',', fmt='%s', header="ClusterID")

In [ ]:
#load all individual feature files
#product.csv = all product-centric features
#user.csv = all reviewer-centric features
#review_features.csv = all review centric features

product = pd.read_csv('product.csv')
print(product)

user = pd.read_csv('user.csv')
print(user)

reviews = pd.read_csv('review_features.csv')
print(reviews)



In [ ]:
#merge all features
#create dictionary of product values
productdct = {}

for index, row in product.iterrows():
  id = int(row['ProductID'])
  if id not in productdct:
    productdct[id] = []
  productdct[id].append(row['MeanRating'])
  productdct[id].append(row['numberOfReviews'])

print(productdct)



In [ ]:
#merge all features
#create all product data
productAllData = []

for index,row in reviews.iterrows():
  id = int(row['productid'])
  productAllData.append(productdct[id])

np.savetxt('productAllData.csv', productAllData, delimiter=',', fmt='%s', header="meanRating,numberOfReviews")

In [ ]:
#merge all features
#create dictionary of user values
userdct = {}

for index, row in user.iterrows():
  id = int(row['userID'])
  if id not in userdct:
    userdct[id] = []
  userdct[id].append(row['avgNoOfWords'])
  userdct[id].append(row['totalReviews'])
  userdct[id].append(row['isSingleton'])
  userdct[id].append(row['posReviewRatio'])
  userdct[id].append(row['negReviewRatio'])
  userdct[id].append(row['avgNoOfReviews'])
  userdct[id].append(row['maxNoOfReviews'])
  userdct[id].append(row['dateVariance'])

In [ ]:
#merge all features
#create all user data
userAllData = []

for index,row in reviews.iterrows():
  id = int(row['userid'])
  userAllData.append(userdct[id])

np.savetxt('userAllData.csv', userAllData, delimiter=',', fmt='%s', header="avgNoOfWords,totalReviews,isSingleton,posReviewRatio,negReviewRatio,avgNoOfReviews,maxNoOfReviews,dateVariance")

In [ ]:
#export labels for all review in a separate file

file2 = open("metadata")
labels = []
pos, neg = 0, 0
for row in file2:
    tr = re.split(r'\t+', row)
    if tr[3] == "1":
        labels.append(0)
        pos +=1 
    else:
        labels.append(1)
        neg += 1

print(pos)
#np.savetxt('labels.csv', labels, delimiter=',', fmt='%s', header="Label")

In [ ]:
np.savetxt('labels.csv', labels, delimiter=',', fmt='%s', header="Label")

In [ ]:
#load all features

X = pd.read_csv('featuresAll.csv')
print(X.head())
X = X.to_numpy()

#load all labels
Y = pd.read_csv('labels.csv')
print(Y.head())
y = Y['label']

In [ ]:
#perform classification using - random forest classifier
#all records are scalled and this uses random sampling

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=53)

from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=56)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(y_pred)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(accuracy_score(y_test, y_pred))

In [ ]:
#perform classification using - random forest classifier
#all records are scalled and this uses stratifiedkfold sampling

from sklearn.model_selection import StratifiedKFold
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


sc = StandardScaler()
skf = StratifiedKFold(n_splits=5)
skf.get_n_splits(X, y)
clf = RandomForestClassifier(n_estimators=50, random_state=56)

for train_ix, test_ix in skf.split(X, y):
  train_X, test_X = X[train_ix], X[test_ix]
  train_y, test_y = y[train_ix], y[test_ix]
  train_0, train_1 = len(train_y[train_y==0]), len(train_y[train_y==1])
  test_0, test_1 = len(test_y[test_y==0]), len(test_y[test_y==1])
  print('>Train: 0=%d, 1=%d, Test: 0=%d, 1=%d' % (train_0, train_1, test_0, test_1))

  train_X = sc.fit_transform(train_X)
  test_X = sc.transform(test_X)
  clf.fit(train_X, train_y)
  y_pred = clf.predict(test_X)
  print(confusion_matrix(test_y,y_pred))
  print(classification_report(test_y,y_pred))
  print(accuracy_score(test_y, y_pred))
  

In [ ]:
#perform classification using - naive bayes classifier
#all records are scalled and this uses stratifiedkfold sampling

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


sc = StandardScaler()
skf = StratifiedKFold(n_splits=5)
skf.get_n_splits(X, y)
clf =  GaussianNB()

for train_ix, test_ix in skf.split(X, y):
  train_X, test_X = X[train_ix], X[test_ix]
  train_y, test_y = y[train_ix], y[test_ix]
  train_0, train_1 = len(train_y[train_y==0]), len(train_y[train_y==1])
  test_0, test_1 = len(test_y[test_y==0]), len(test_y[test_y==1])
  print('>Train: 0=%d, 1=%d, Test: 0=%d, 1=%d' % (train_0, train_1, test_0, test_1))

  train_X = sc.fit_transform(train_X)
  test_X = sc.transform(test_X)
  clf.fit(train_X, train_y)
  y_pred = clf.predict(test_X)
  print(confusion_matrix(test_y,y_pred))
  print(classification_report(test_y,y_pred))
  print(accuracy_score(test_y, y_pred))